In [1]:
# Step 1: Import Qiskit libraries, Aer, and noise-model tools

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
import numpy as np
import plotly.graph_objects as go

In [2]:
# Step 2: Set common parameters

N_BITS = 200
rng = np.random.default_rng()   # no seed = fully random each run

In [3]:
# Step 3: Build a noise model from a real IBM backend calibration snapshot

from qiskit_ibm_runtime.fake_provider import FakeManilaV2

backend1 = FakeManilaV2()
noise_model_1 = NoiseModel.from_backend(backend1)

print("Noise model 1 (FakeManilaV2) loaded.")
print(noise_model_1)

Noise model 1 (FakeManilaV2) loaded.
NoiseModel:
  Basis gates: ['cx', 'delay', 'for_loop', 'id', 'if_else', 'measure', 'reset', 'rz', 'switch_case', 'sx', 'x']
  Instructions with noise: ['reset', 'sx', 'x', 'id', 'cx', 'measure']
  Qubits with noise: [0, 1, 2, 3, 4]
  Specific qubit errors: [('reset', (0,)), ('reset', (1,)), ('reset', (2,)), ('reset', (3,)), ('reset', (4,)), ('sx', (0,)), ('sx', (1,)), ('sx', (2,)), ('sx', (3,)), ('sx', (4,)), ('x', (0,)), ('x', (1,)), ('x', (2,)), ('x', (3,)), ('x', (4,)), ('id', (0,)), ('id', (1,)), ('id', (2,)), ('id', (3,)), ('id', (4,)), ('cx', (0, 1)), ('cx', (1, 0)), ('cx', (1, 2)), ('cx', (2, 1)), ('cx', (2, 3)), ('cx', (3, 2)), ('cx', (3, 4)), ('cx', (4, 3)), ('measure', (0,)), ('measure', (1,)), ('measure', (2,)), ('measure', (3,)), ('measure', (4,))]


In [4]:
# Step 4: Build a second noise model from a different IBM backend snapshot

from qiskit_ibm_runtime.fake_provider import FakeLimaV2

backend2 = FakeLimaV2()
noise_model_2 = NoiseModel.from_backend(backend2)

print("Noise model 2 (FakeLimaV2) loaded.")
print(noise_model_2)

Noise model 2 (FakeLimaV2) loaded.
NoiseModel:
  Basis gates: ['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']
  Instructions with noise: ['reset', 'sx', 'x', 'id', 'cx', 'measure']
  Qubits with noise: [0, 1, 2, 3, 4]
  Specific qubit errors: [('reset', (0,)), ('reset', (1,)), ('reset', (2,)), ('reset', (3,)), ('reset', (4,)), ('sx', (0,)), ('sx', (1,)), ('sx', (2,)), ('sx', (3,)), ('sx', (4,)), ('x', (0,)), ('x', (1,)), ('x', (2,)), ('x', (3,)), ('x', (4,)), ('id', (0,)), ('id', (1,)), ('id', (2,)), ('id', (3,)), ('id', (4,)), ('cx', (0, 1)), ('cx', (1, 0)), ('cx', (1, 2)), ('cx', (1, 3)), ('cx', (2, 1)), ('cx', (3, 1)), ('cx', (3, 4)), ('cx', (4, 3)), ('measure', (0,)), ('measure', (1,)), ('measure', (2,)), ('measure', (3,)), ('measure', (4,))]


In [5]:
# Step 5: Run BB84 circuits under a given noise model, collect Bob's results

def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 'X':
        qc.h(0)
    return qc

def measure_qubit(qc, basis):
    if basis == 'X':
        qc.h(0)
    qc.measure(0, 0)
    return qc

def run_bb84_noisy(alice_bits, alice_bases, bob_bases, noise_model):
    sim_noisy = AerSimulator(noise_model=noise_model)
    bob_results = []
    for i in range(len(alice_bits)):
        qc = encode_qubit(alice_bits[i], alice_bases[i])
        qc = measure_qubit(qc, bob_bases[i])
        result = sim_noisy.run(qc, shots=1, memory=True).result()
        outcome = int(result.get_memory()[0])
        bob_results.append(outcome)
    return np.array(bob_results)

# Alice and Bob random bits/bases (same for both noise models, for fair comparison)
alice_bits = rng.integers(0, 2, N_BITS)
alice_bases = rng.choice(['Z', 'X'], size=N_BITS)
bob_bases = rng.choice(['Z', 'X'], size=N_BITS)

bob_results_noise1 = run_bb84_noisy(alice_bits, alice_bases, bob_bases, noise_model_1)
bob_results_noise2 = run_bb84_noisy(alice_bits, alice_bases, bob_bases, noise_model_2)

print("Bob's results under noise model 1 (first 10):", bob_results_noise1[:10])
print("Bob's results under noise model 2 (first 10):", bob_results_noise2[:10])

Bob's results under noise model 1 (first 10): [0 0 1 1 0 1 0 0 0 1]
Bob's results under noise model 2 (first 10): [1 0 1 1 0 1 0 0 1 1]


In [6]:
# Step 6: Run LM05 circuits under a given noise model, collect CM/MM results

def prepare_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 'X':
        qc.h(0)
    return qc

def alice_cm_measure(qc, basis):
    if basis == 'X':
        qc.h(0)
    qc.measure(0, 0)
    return qc

def alice_mm_encode(qc, message_bit):
    if message_bit == 1:
        qc.y(0)
    return qc

def bob_final_measure(qc, basis):
    if basis == 'X':
        qc.h(0)
    qc.measure(0, 0)
    return qc

def run_lm05_noisy(bob_bits, bob_bases, alice_modes, alice_cm_bases, alice_message_bits, noise_model):
    sim_noisy = AerSimulator(noise_model=noise_model)
    cm_results = []
    mm_bob_decoded = []

    for i in range(len(bob_bits)):
        qc = prepare_qubit(bob_bits[i], bob_bases[i])

        if alice_modes[i] == 'CM':
            qc = alice_cm_measure(qc, alice_cm_bases[i])
            result = sim_noisy.run(qc, shots=1, memory=True).result()
            outcome = int(result.get_memory()[0])
            cm_results.append(outcome)
        else:  # MM
            qc = alice_mm_encode(qc, alice_message_bits[i])
            qc = bob_final_measure(qc, bob_bases[i])
            result = sim_noisy.run(qc, shots=1, memory=True).result()
            outcome = int(result.get_memory()[0])
            decoded_bit = 0 if outcome == bob_bits[i] else 1
            mm_bob_decoded.append(decoded_bit)

    return np.array(cm_results), np.array(mm_bob_decoded)

# Bob and Alice random values (same for both noise models, for fair comparison)
bob_bits_lm05 = rng.integers(0, 2, N_BITS)
bob_bases_lm05 = rng.choice(['Z', 'X'], size=N_BITS)
alice_modes = rng.choice(['CM', 'MM'], size=N_BITS)
alice_cm_bases = rng.choice(['Z', 'X'], size=N_BITS)
alice_message_bits = rng.integers(0, 2, N_BITS)

cm_results_noise1, mm_decoded_noise1 = run_lm05_noisy(
    bob_bits_lm05, bob_bases_lm05, alice_modes, alice_cm_bases, alice_message_bits, noise_model_1)

cm_results_noise2, mm_decoded_noise2 = run_lm05_noisy(
    bob_bits_lm05, bob_bases_lm05, alice_modes, alice_cm_bases, alice_message_bits, noise_model_2)

print("LM05 CM results under noise model 1 (first 10):", cm_results_noise1[:10])
print("LM05 CM results under noise model 2 (first 10):", cm_results_noise2[:10])

LM05 CM results under noise model 1 (first 10): [1 0 1 0 1 0 0 1 1 0]
LM05 CM results under noise model 2 (first 10): [1 1 1 0 0 1 0 1 1 0]


In [7]:
# Step 7: Sifting and QBER calculation for BB84 (noisy) — for both noise models

def compute_bb84_qber(alice_bits, alice_bases, bob_bases, bob_results):
    sift_mask = alice_bases == bob_bases
    alice_sifted = alice_bits[sift_mask]
    bob_sifted = bob_results[sift_mask]
    mismatches = np.sum(alice_sifted != bob_sifted)
    qber = mismatches / len(alice_sifted)
    return qber, len(alice_sifted)

qber_bb84_noise1, sifted_len_bb84_noise1 = compute_bb84_qber(alice_bits, alice_bases, bob_bases, bob_results_noise1)
qber_bb84_noise2, sifted_len_bb84_noise2 = compute_bb84_qber(alice_bits, alice_bases, bob_bases, bob_results_noise2)

print(f"BB84 QBER (Noise Model 1 - FakeManilaV2): {qber_bb84_noise1:.4f}, sifted key length: {sifted_len_bb84_noise1}")
print(f"BB84 QBER (Noise Model 2 - FakeLimaV2)  : {qber_bb84_noise2:.4f}, sifted key length: {sifted_len_bb84_noise2}")

BB84 QBER (Noise Model 1 - FakeManilaV2): 0.0500, sifted key length: 100
BB84 QBER (Noise Model 2 - FakeLimaV2)  : 0.0800, sifted key length: 100


In [8]:
# Step 8: Control-Mode QBER calculation for LM05 (noisy) — for both noise models

def compute_lm05_qber(bob_bits, bob_bases, alice_modes, alice_cm_bases, cm_results):
    cm_mask = alice_modes == 'CM'
    cm_bob_bits = bob_bits[cm_mask]
    cm_bob_bases = bob_bases[cm_mask]
    cm_alice_bases = alice_cm_bases[cm_mask]

    basis_match_mask = cm_alice_bases == cm_bob_bases
    cm_checkable_bob = cm_bob_bits[basis_match_mask]
    cm_checkable_alice = cm_results[basis_match_mask]

    mismatches = np.sum(cm_checkable_alice != cm_checkable_bob)
    qber = mismatches / len(cm_checkable_alice)
    return qber, len(cm_checkable_alice)

qber_lm05_noise1, cm_checkable_len_noise1 = compute_lm05_qber(
    bob_bits_lm05, bob_bases_lm05, alice_modes, alice_cm_bases, cm_results_noise1)

qber_lm05_noise2, cm_checkable_len_noise2 = compute_lm05_qber(
    bob_bits_lm05, bob_bases_lm05, alice_modes, alice_cm_bases, cm_results_noise2)

print(f"LM05 QBER (Noise Model 1 - FakeManilaV2): {qber_lm05_noise1:.4f}, checkable CM rounds: {cm_checkable_len_noise1}")
print(f"LM05 QBER (Noise Model 2 - FakeLimaV2)  : {qber_lm05_noise2:.4f}, checkable CM rounds: {cm_checkable_len_noise2}")

LM05 QBER (Noise Model 1 - FakeManilaV2): 0.0652, checkable CM rounds: 46
LM05 QBER (Noise Model 2 - FakeLimaV2)  : 0.0652, checkable CM rounds: 46


In [9]:
# Step 9: Compare BB84 vs LM05 QBER across noise models

print(f"{'Noise Model':<20}{'BB84 QBER':<15}{'LM05 QBER':<15}")
print("-" * 50)
print(f"{'FakeManilaV2':<20}{qber_bb84_noise1:<15.4f}{qber_lm05_noise1:<15.4f}")
print(f"{'FakeLimaV2':<20}{qber_bb84_noise2:<15.4f}{qber_lm05_noise2:<15.4f}")

Noise Model         BB84 QBER      LM05 QBER      
--------------------------------------------------
FakeManilaV2        0.0500         0.0652         
FakeLimaV2          0.0800         0.0652         


In [10]:
# Step 10: Predict expected QBER trend before real hardware (Level 3)

avg_qber_bb84 = np.mean([qber_bb84_noise1, qber_bb84_noise2])
avg_qber_lm05 = np.mean([qber_lm05_noise1, qber_lm05_noise2])

print(f"Average BB84 QBER across noise models: {avg_qber_bb84:.4f}")
print(f"Average LM05 QBER across noise models: {avg_qber_lm05:.4f}")

if avg_qber_lm05 > avg_qber_bb84:
    print("Prediction: LM05 likely to show HIGHER QBER on real hardware (round-trip noise exposure).")
elif avg_qber_lm05 < avg_qber_bb84:
    print("Prediction: LM05 likely to show LOWER QBER on real hardware.")
else:
    print("Prediction: Both protocols likely to show similar QBER on real hardware.")

Average BB84 QBER across noise models: 0.0650
Average LM05 QBER across noise models: 0.0652
Prediction: LM05 likely to show HIGHER QBER on real hardware (round-trip noise exposure).


In [11]:
# Step 11: Visualize comparison — QBER across noise models, BB84 vs LM05

noise_models_labels = ['FakeManilaV2', 'FakeLimaV2']
bb84_qbers = [qber_bb84_noise1, qber_bb84_noise2]
lm05_qbers = [qber_lm05_noise1, qber_lm05_noise2]

fig = go.Figure()
fig.add_trace(go.Bar(name='BB84', x=noise_models_labels, y=bb84_qbers, marker_color='#185FA5'))
fig.add_trace(go.Bar(name='LM05', x=noise_models_labels, y=lm05_qbers, marker_color='#6E2E8C'))

fig.update_layout(
    barmode='group',
    title='QBER Comparison Across Noise Models: BB84 vs LM05',
    xaxis_title='Noise Model',
    yaxis_title='QBER',
)

fig.show()

In [12]:
# Step 12: Calculate SKR for BB84 and LM05 under both noise models

def binary_entropy(p):
    if p == 0 or p == 1:
        return 0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

f_EC = 1.16   # typical error-correction inefficiency factor

def compute_skr(qber, sifted_len):
    skr_fraction = max(1 - f_EC * binary_entropy(qber) - binary_entropy(qber), 0)
    return skr_fraction * sifted_len

skr_bb84_noise1 = compute_skr(qber_bb84_noise1, sifted_len_bb84_noise1)
skr_bb84_noise2 = compute_skr(qber_bb84_noise2, sifted_len_bb84_noise2)

skr_lm05_noise1 = compute_skr(qber_lm05_noise1, len(alice_message_bits[alice_modes == 'MM']))
skr_lm05_noise2 = compute_skr(qber_lm05_noise2, len(alice_message_bits[alice_modes == 'MM']))

print(f"{'Noise Model':<20}{'BB84 SKR (bits)':<20}{'LM05 SKR (bits)':<20}")
print("-" * 60)
print(f"{'FakeManilaV2':<20}{skr_bb84_noise1:<20.1f}{skr_lm05_noise1:<20.1f}")
print(f"{'FakeLimaV2':<20}{skr_bb84_noise2:<20.1f}{skr_lm05_noise2:<20.1f}")

Noise Model         BB84 SKR (bits)     LM05 SKR (bits)     
------------------------------------------------------------
FakeManilaV2        38.1                27.6                
FakeLimaV2          13.1                27.6                
